# Baseline Recommender 
This notebook builds the first CalCourse recommendation baseline using TF-IDF and cosine similarity to rank eligible courses by relevance to a student profile.

In [1]:
import pandas as pd 
import networkx as nx 

courses = pd.read_csv(
    "../data/processed/recommendable_courses_fall_2026.csv"
)

prereqs = pd.read_csv(
    "../data/processed/prerequisites_fall_2026.csv"
)

In [2]:
prereqs["course"] = (
    prereqs["subject"] + " " + prereqs["course_number"].astype(str)
)

prereqs["prerequisite"] = (
    prereqs["prereq_subject"] + " " + prereqs["prereq_number"].astype(str)
)

G = nx.DiGraph()

for _, row in prereqs.iterrows():
    G.add_edge(row["prerequisite"], row["course"])

In [3]:
completed_courses = {
    "DATA C8",
    "COMPSCI 61A",
    "MATH 1A",
    "MATH 1B"
}

student_interests = """
machine learning statistics data science product analytics
"""

In [4]:
def get_prerequisites(course):
    if course not in G:
        return set()

    return set(G.predecessors(course))


def missing_prerequisites(course, completed_courses):
    return get_prerequisites(course) - completed_courses

In [5]:
courses["course"] = (
    courses["subject"] + " " + courses["course_number"].astype(str)
)

courses["missing_prereqs"] = courses["course"].apply(
    lambda course: missing_prerequisites(course, completed_courses)
)

eligible_courses = courses[
    courses["missing_prereqs"].apply(len) == 0
].copy()

eligible_courses.shape

(1725, 10)

In [6]:
eligible_courses["text"] = (
    eligible_courses["title"].fillna("") + " " +
    eligible_courses["description"].fillna("")
)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
vectorizer = TfidfVectorizer(
    stop_words="english"
)

In [10]:
course_matrix = vectorizer.fit_transform(
    eligible_courses["text"]
)

In [11]:
student_vector = vectorizer.transform(
    [student_interests]
)

In [12]:
similarity_scores = cosine_similarity(
    student_vector,
    course_matrix
).flatten()

In [13]:
eligible_courses["similarity_score"] = similarity_scores

In [14]:
recommendations = eligible_courses.sort_values(
    "similarity_score",
    ascending=False
)

In [15]:
recommendations[
    ["course", "title", "similarity_score"]
].head(15)

,course,title,similarity_score
1033,INDENG 142A,Introduction to Machine Learning and Data Anal...,0.382353
453,DATA 188,Advanced Data Science Connector,0.302737
439,CYPLAN 101,Introduction to Urban Data Analytics,0.297465
460,STS C104D,Human Contexts and Ethics of Data - DATA/Histo...,0.292148
1916,STAT 157,Seminar on Topics in Probability and Statistics,0.287889
454,DATA 36,Data Scholars Seminar,0.275175
1918,STAT 197,Field Study in Statistics,0.261171
434,STAT C8,Foundations of Data Science,0.210679
464,DATA C6,Introduction to Computational Thinking with Da...,0.203917
458,DATA 94,Special Topics in Data Science,0.202272


### Baseline Observations
The TF-IDF baseline produces some relevant results but tends to over-rank courses with strong keyword overlap, including seminars and special topics. It doesn't capture deeper semantic relevance or distinguish foundational courses from niche offerings. 